<a href="https://colab.research.google.com/github/Antibote/pinn-eos-aluminum/blob/main/notebooks/section3_pinn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Раздел 3. PINN: поиск гиперпараметров и финальная модель

Порядок: **раздел 0** → (если в `artifacts/` нет `split.npz` и `normalization.json` — запустить 1.3 в общем блокноте) → 3.0 → 3.1 → 3.2 → 3.3 → 3.4.

Поиск идёт поэтапно, лучшее на каждом этапе выбирается автоматически по val (`pn.pick_best`):
- **3.1 Этап A** — функция потерь по данным: MSE, L1, Huber, log-cosh;
- **3.2 Этап B** — активация: SiLU, Tanh, GELU, SELU, Softplus, Sin;
- **3.3 Этап C** — расчёт градиента со штрафом согласованности: fixed, adaptive (Wang и др., 2021), PCGrad (Yu и др., 2020);
- **3.4** — финальное обучение лучшей конфигурации (256×4, 30 000 шагов), сравнение с бейзлайном и Vanilla на test.

Правило отбора: среди вариантов с нарушениями устойчивости (∂P/∂ρ > 0, ∂e/∂T ≥ 0, c_s² > 0) < 0.1% точек — минимальный score = медианы ошибок P и e + 0.1·их p99 (%).
Код — `src/eos/pinn.py`. Все результаты сохраняются в `artifacts/` и отправляются на GitHub через `save_to_repo`.

## 0. Подготовка (общая, запускать каждому)

In [1]:
import torch
print('torch', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'нет — включите GPU в настройках среды')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch 2.11.0+cu130
GPU: Tesla T4


In [ ]:
# Репозиторий: код (src/eos) и папка artifacts — сюда сохраняются все результаты.
# На GitHub их отправляет save_to_repo('что сохранили') — см. ячейку ниже.
import json, subprocess, sys, urllib.parse, urllib.request, zipfile
from pathlib import Path

REPO = Path('pinn-eos-aluminum')
REPO_URL = 'https://github.com/Antibote/pinn-eos-aluminum.git'
_TOKEN = None


def git(*args):
    """git в папке репозитория. Токен в сообщениях об ошибке скрыт."""
    r = subprocess.run(['git', '-C', str(REPO), *args], capture_output=True, text=True)
    if r.returncode:
        msg = (r.stderr or r.stdout).strip()
        raise RuntimeError(f'git {args[0]}: ' + (msg.replace(_TOKEN, '***') if _TOKEN else msg))
    return r.stdout


if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', REPO_URL, str(REPO)], check=True)
git('config', 'user.name', 'Colab')
git('config', 'user.email', 'colab@users.noreply.github.com')
git('pull', '-q', '--rebase', '--autostash')
sys.path.insert(0, str(REPO / 'src'))

ARTIFACTS = REPO / 'artifacts'
FIGURES = ARTIFACTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
print('Артефакты:', ARTIFACTS.resolve())


In [ ]:
def save_to_repo(message):
    """Коммитит папку artifacts и отправляет её на GitHub.

    Нужен секрет Colab GITHUB_TOKEN (значок ключа слева) с доступом к репозиторию.

    Args:
        message: что сохранили, например 'Бейзлайн: метрики и карта ошибок'.
    """
    global _TOKEN
    from google.colab import userdata
    _TOKEN = userdata.get('GITHUB_TOKEN')

    big = [str(p) for p in ARTIFACTS.rglob('*') if p.is_file() and p.stat().st_size > 50 * 2**20]
    if big:
        raise RuntimeError(f'Файлы больше 50 МБ GitHub не примет: {big}')
    git('add', 'artifacts')
    changed = git('diff', '--cached', '--name-only', '--', 'artifacts').split()
    if not changed:
        print('Нечего сохранять: artifacts не изменилась')
        return

    # Автор коммита — владелец токена
    req = urllib.request.Request('https://api.github.com/user', headers={'Authorization': f'token {_TOKEN}'})
    me = json.load(urllib.request.urlopen(req))
    author = f"{me.get('name') or me['login']} <{me['id']}+{me['login']}@users.noreply.github.com>"

    git('commit', '-q', '-m', message, '--author', author)
    url = REPO_URL.replace('https://', f'https://x-access-token:{_TOKEN}@')
    git('pull', '-q', '--rebase', url, 'main')
    git('push', '-q', url, 'HEAD:main')
    print('На GitHub:', git('log', '-1', '--format=%h %s').strip())
    print('Файлы:', *changed, sep='\n  ')


In [ ]:
# Данные с Яндекс Диска (в репозиторий не попадают)
DATA = REPO / 'data'
if not list(DATA.glob('Obl*.txt')):
    api = 'https://cloud-api.yandex.net/v1/disk/public/resources/download?' + urllib.parse.urlencode(
        {'public_key': 'https://disk.yandex.ru/d/iUSbYrJcsbXSbw', 'path': '/Материалы к хакатону.zip'})
    href = json.load(urllib.request.urlopen(api))['href']
    urllib.request.urlretrieve(href, 'data.zip')
    zipfile.ZipFile('data.zip').extractall(DATA)
print(len(list(DATA.glob('Obl*.txt'))), 'файлов данных в', DATA)


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from eos.data import load_all, region_grid

df = load_all(DATA)
print(df.shape)
df.head()

(1130053, 5)


,rho,T,P,E,region
0,0.001000,0.1,0.000515,-46.797895,Obl01
1,0.001030,0.1,0.000540,-46.790300,Obl01
2,0.001060,0.1,0.000569,-46.782552,Obl01
3,0.001091,0.1,0.000597,-46.774662,Obl01
4,0.001124,0.1,0.000626,-46.766630,Obl01


---
## 3. PINN и физические ограничения

### 3.0 Подготовка

In [ ]:
# 3.0 Подготовка: данные на GPU, val для отбора, согласованность самой таблицы (код — src/eos/pinn.py)
import importlib, json, torch
import eos.normalize as nz, eos.model as em, eos.pinn as pn, eos.metrics as mt
for m in (nz, em, pn, mt):
    importlib.reload(m)
pd.set_option('display.width', 250)

norm = nz.EOSNormalizer.load(ARTIFACTS / 'normalization.json')
split = dict(np.load(ARTIFACTS / 'split.npz'))
val_df, test_df = df.iloc[split['val']], df.iloc[split['test']]

def to_t(a):
    return torch.tensor(np.asarray(a), dtype=torch.float32, device=DEVICE)

trainer = pn.Trainer(*(to_t(df[c].values) for c in ['rho', 'T', 'P', 'e']),
                     idx_train=torch.tensor(split['train'], device=DEVICE),
                     idx_val=torch.tensor(split['val'], device=DEVICE),
                     w_train=to_t(em.region_weights(df.iloc[split['train']])))

# Настройки поиска. Каждый вариант учится STEPS_SEARCH шагов; отбор — только по val
STEPS_SEARCH = 5_000
BASE = dict(hidden=128, layers=4, activation='silu', loss='mse', grad='fixed',
            lam={'mono_rho': 1, 'mono_T': 1})          # штрафы устойчивости: ∂P/∂ρ > 0, ∂e/∂T ≥ 0
COLS = ['score', 'err_P_median_%', 'err_P_p99_%', 'err_e_median_%', 'err_e_p99_%',
        'viol_dP_drho_%', 'viol_de_dT_%', 'viol_cs2_%', 'cons_median', 'time_s']
(ARTIFACTS / 'pinn_search').mkdir(exist_ok=True)

# Насколько согласована сама таблица: (∂e/∂ρ)_T = (P − T ∂P/∂T)/ρ².
# ratio_median = 1 — согласовано; 0.5 — энергия меняется с ρ вдвое медленнее, чем требует термодинамика
cons_table = pn.table_consistency(df)
cons_table.to_csv(ARTIFACTS / 'table_consistency.csv', index=False)
print(cons_table.to_string(index=False, float_format=lambda v: f'{v:.3g}'))

### 3.1 Этап A: функция потерь

In [ ]:
# 3.1 Этап A: функция потерь по данным (активация SiLU, градиент fixed)
#   mse — квадрат ошибки; l1 — модуль; huber и logcosh — квадрат для малых ошибок, модуль для больших
search_A, _, hist_A = pn.search(trainer, norm, BASE, {f'loss={k}': {'loss': k} for k in pn.DATA_LOSSES},
                                val_df, STEPS_SEARCH, DEVICE)
search_A.to_csv(ARTIFACTS / 'pinn_search' / 'A_loss.csv')
display(search_A[COLS].round(4))
BASE['loss'] = search_A.loc[pn.pick_best(search_A), 'loss']
print('Лучшая функция потерь:', BASE['loss'])

### 3.2 Этап B: функция активации

In [ ]:
# 3.2 Этап B: функция активации (с лучшей функцией потерь из этапа A)
# Для PINN нужна гладкая активация: производные сети входят в loss.
# SELU — с изломом производной в нуле, Sin — периодическая (SIREN), остальные гладкие
search_B, _, hist_B = pn.search(trainer, norm, BASE,
                                {f'act={k}': {'activation': k} for k in em.ACTIVATIONS if k != 'silu'},
                                val_df, STEPS_SEARCH, DEVICE)
silu_row = search_A.loc[[f"loss={BASE['loss']}"]].rename(index=lambda _: 'act=silu').assign(activation='silu')
search_B = pd.concat([silu_row, search_B])                # SiLU уже обучен на этапе A
search_B.to_csv(ARTIFACTS / 'pinn_search' / 'B_activation.csv')
display(search_B[COLS].round(4))
BASE['activation'] = search_B.loc[pn.pick_best(search_B), 'activation']
print('Лучшая активация:', BASE['activation'])

### 3.3 Этап C: расчёт градиента

In [ ]:
# 3.3 Этап C: как объединять градиенты данных и физики — теперь со штрафом согласованности.
# Таблица несогласована (3.0), поэтому этот штраф спорит с данными; вопрос — какой способ
# расчёта градиента позволит его учесть без потери точности:
#   fixed    — loss = данные + Σ λ·штраф, один backward;
#   adaptive — λ подстраиваются по нормам градиентов (Wang, Teng, Perdikaris, 2021);
#   pcgrad   — при конфликте направлений убирается проекция одного градиента на другой (Yu et al., 2020)
LAM_CONS = {'mono_rho': 1, 'mono_T': 1, 'cons': 0.01}
search_C, _, hist_C = pn.search(trainer, norm, BASE,
                                {f'grad={k}+cons': {'grad': k, 'lam': LAM_CONS} for k in pn.GRAD_MODES},
                                val_df, STEPS_SEARCH, DEVICE)
best_B = search_B.loc[[pn.pick_best(search_B)]].rename(index=lambda _: 'grad=fixed (без cons)')
search_C = pd.concat([best_B.assign(grad='fixed', lam=json.dumps(BASE['lam'])), search_C])
search_C.to_csv(ARTIFACTS / 'pinn_search' / 'C_grad.csv')
display(search_C[COLS].round(4))
print('Адаптивные множители λ:', hist_C['grad=adaptive+cons']['adapt'])

best_C = pn.pick_best(search_C)
BASE['grad'] = search_C.loc[best_C, 'grad']
if best_C != 'grad=fixed (без cons)':
    BASE['lam'] = LAM_CONS
print('Лучший способ:', best_C, '→', BASE)

# Итог поиска: все этапы на одном графике (меньше score — лучше)
allv = pd.concat([search_A.assign(этап='A: loss'), search_B.assign(этап='B: активация'),
                  search_C.assign(этап='C: градиент')])
fig, ax = plt.subplots(figsize=(12, 4.5))
colors = {'A: loss': 'C0', 'B: активация': 'C1', 'C: градиент': 'C2'}
ax.bar(range(len(allv)), allv['score'], color=[colors[s] for s in allv['этап']])
ax.set_xticks(range(len(allv)), allv.index, rotation=45, ha='right')
ax.set(ylabel='score (медианы + 0.1·p99 ошибок P и e, %)', title='Поиск гиперпараметров PINN (val)')
for s, c in colors.items():
    ax.bar(0, 0, color=c, label=s)
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES / 'pinn_search.png', dpi=130, bbox_inches='tight')
plt.show()
save_to_repo('3.1–3.3: поиск гиперпараметров PINN')

### 3.4 Финальный PINN и сравнение

In [ ]:
# 3.4 Финальный PINN: лучшие настройки из 3.1–3.3, сеть шире (как Vanilla в разделе 2) и обучение дольше
FINAL = {**BASE, 'hidden': 256, 'layers': 4}
STEPS_FINAL = 30_000
print('Конфигурация:', FINAL)

torch.manual_seed(0)
pinn = em.EOSNet(norm, hidden=FINAL['hidden'], layers=FINAL['layers'], activation=FINAL['activation']).to(DEVICE)
hist = trainer.fit(pinn, lam=FINAL['lam'], steps=STEPS_FINAL, loss=FINAL['loss'], grad=FINAL['grad'])
torch.save({'state_dict': pinn.state_dict(), 'config': pinn.config, 'train': FINAL, 'steps': STEPS_FINAL},
           ARTIFACTS / 'pinn.pt')


class TorchEOS:
    """Обёртка для замера скорости: float32 на DEVICE, как в рабочем инференсе."""

    def __init__(self, model):
        self.model = model.eval()

    @torch.no_grad()
    def predict(self, rho, T):
        P, e = self.model(to_t(rho), to_t(T))
        return P.cpu().numpy(), e.cpu().numpy()


# Метрики на test — той же функцией, что у бейзлайна и Vanilla
import eos.baseline as bl
metrics, per_point = mt.evaluate(pn.numpy_predict(pinn), test_df, f"PINN ({FINAL['activation']}, {FINAL['loss']}, {FINAL['grad']})")
metrics['time_1e5_s'] = bl.benchmark(TorchEOS(pinn), n=100_000)
metrics['size_MB'] = sum(p.numel() for p in pinn.parameters()) * 4 / 2**20
metrics.update({f'autograd_{k}': v for k, v in pn.physics_report(pinn, n=100_000).items()})
with open(ARTIFACTS / 'metrics_pinn.json', 'w') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

rows = [metrics]
for name in ['metrics_vanilla.json', 'metrics_baseline.json']:
    if (ARTIFACTS / name).exists():
        with open(ARTIFACTS / name) as f:
            rows.insert(0, json.load(f))
keys = ['err_P_median_%', 'err_P_p99_%', 'err_P_max_%', 'err_e_median_%', 'err_e_p99_%', 'err_e_max_%',
        'viol_dP_drho_%', 'viol_de_dT_%', 'viol_dP_dT_%', 'time_1e5_s', 'size_MB']
compare = pd.DataFrame(rows).set_index('model')
compare.to_csv(ARTIFACTS / 'metrics_compare.csv')
display(compare[keys].T.round(4))
print('Физика PINN через autograd (100 000 случайных точек):')
print(pd.Series({k: v for k, v in metrics.items() if k.startswith('autograd_')}).round(4).to_string())

# Кривая обучения
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(hist['step'], hist['val']); ax[0].set(yscale='log', xlabel='шаг', ylabel='MSE по данным (val)', title='Точность')
ax[1].plot(hist['step'], hist['phys']); ax[1].set(yscale='symlog', xlabel='шаг', ylabel='λ · штраф (train)', title='Физика')
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES / 'pinn_learning_curve.png', dpi=130, bbox_inches='tight')
plt.show()

# Карта ошибок на (ρ, T)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for a, col, title in [(ax[0], 'err_P', 'ошибка P, %'), (ax[1], 'err_e', 'ошибка e, %')]:
    sc = a.scatter(per_point['rho'], per_point['T'], c=np.log10(per_point[col] * 100 + 1e-6), s=0.3,
                   cmap='viridis', vmin=-3, vmax=2, rasterized=True)
    a.set(xscale='log', yscale='log', xlabel='ρ, г/см³', ylabel='T, K', title=f'PINN: {title}')
    plt.colorbar(sc, ax=a, label='log10(ошибка, %)')
plt.tight_layout()
plt.savefig(FIGURES / 'pinn_errors.png', dpi=130, bbox_inches='tight')
plt.show()


def section(col, v):
    """Точки таблицы при col ≈ v: в каждой области свой ближайший узел (сетки разные)."""
    parts = []
    for _, g in df.groupby('region'):
        u = np.unique(g[col])
        k = np.abs(np.log(u / v)).argmin()
        if abs(np.log(u[k] / v)) < 0.02:
            parts.append(g[g[col] == u[k]])
    return pd.concat(parts)


# Сечения: таблица (точки) и PINN (линии); внизу — знак производных
predict = pn.numpy_predict(pinn)
fig, ax = plt.subplots(2, 2, figsize=(14, 9))
T_line, rho_line = np.logspace(-1, 7, 600), np.logspace(-3, 4, 600)
for k, rv in enumerate([0.01, 2.7, 100.0]):
    pts, r_ = section('rho', rv), np.full_like(T_line, rv)
    ax[0, 0].scatter(pts['T'], pts['e'], s=3, color=f'C{k}', alpha=0.3)
    ax[0, 0].plot(T_line, predict(r_, T_line)[1], color=f'C{k}', lw=1, label=f'ρ={rv:g}')
    d = pn.log_derivatives(pinn, to_t(r_), to_t(T_line), create_graph=False)
    ax[1, 0].plot(T_line, d['e_t'].detach().cpu().numpy(), color=f'C{k}', lw=1, label=f'ρ={rv:g}')
for k, Tv in enumerate([10.0, 1e3, 1e5]):
    pts, T_ = section('T', Tv), np.full_like(rho_line, Tv)
    ax[0, 1].scatter(pts['rho'], pts['P'], s=3, color=f'C{k}', alpha=0.3)
    ax[0, 1].plot(rho_line, predict(rho_line, T_)[0], color=f'C{k}', lw=1, label=f'T={Tv:g} K')
    d = pn.log_derivatives(pinn, to_t(rho_line), to_t(T_), create_graph=False)
    ax[1, 1].plot(rho_line, d['a'].detach().cpu().numpy(), color=f'C{k}', lw=1, label=f'T={Tv:g} K')
ax[0, 0].set(xscale='log', yscale='symlog', xlabel='T, K', ylabel='e, кДж/г', title='e(T): точки — таблица, линии — PINN')
ax[0, 1].set(xscale='log', yscale='log', xlabel='ρ, г/см³', ylabel='P, ГПа', title='P(ρ): точки — таблица, линии — PINN')
ax[1, 0].set(xscale='log', yscale='symlog', xlabel='T, K', ylabel='∂e/∂lnT', title='∂e/∂lnT ≥ 0 — термическая устойчивость')
ax[1, 1].set(xscale='log', xlabel='ρ, г/см³', ylabel='∂lnP/∂lnρ', title='∂lnP/∂lnρ > 0 — механическая устойчивость')
for a in ax.ravel():
    a.grid(alpha=0.3); a.legend(fontsize=8)
ax[1, 0].axhline(0, color='k', lw=0.8); ax[1, 1].axhline(0, color='k', lw=0.8)
plt.tight_layout()
plt.savefig(FIGURES / 'pinn_sections.png', dpi=130, bbox_inches='tight')
plt.show()

save_to_repo(f"3.4: финальный PINN ({FINAL['activation']}, {FINAL['loss']}, {FINAL['grad']})")